# Minimality analysis

Complement to `decoder_analysis.ipynb`. That notebook tests **sufficiency**: is the posterior mean/precision linearly decodable from `r_hid`? This notebook tests **minimality**, following the distinction raised in Kalburge & Lengyel (2025)'s functional information bottleneck (fIB): has the network actually compressed away everything except what's needed for the posterior, or does `r_hid` still linearly carry the raw input around (a heuristic recoding rather than a genuine sufficient-statistic code)?

We do this by training the same kind of decoder used for opt-precision, but targeting the raw input signal `x_{t-lag}` instead, at several lags. Good posterior decodability *and* good input decodability (especially at short lags) means the code is not minimal.

This notebook is self-contained: it re-collects data from the trained checkpoint (this time keeping the raw input, which `correlation_analysis.ipynb`'s `collect()` does not save), then runs the lag sweep.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %cd /content/kalnet
    %pip install -e .
else:
    # update for your local path
    %cd /home/jacob/kalnet


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from kalnet.task_vec import KalmanFilteringTask
from kalnet.model import KalmanRNN
from kalnet.decoders import (
    LinearDecoder,
    opt_precision,
    pool_timesteps,
    select_trials,
    trial_train_validation_indices,
)
from kalnet.minimality import build_lagged_dataset, center_per_timestep, run_lag_sweep


## Re-collect data, keeping the raw input

Same checkpoint and task conditions as `correlation_analysis.ipynb` / `decoder_analysis.ipynb`, but this `collect()` also stashes `x_mean`: the per-timestep mean input-population response (equivalently the raw scalar measurement, if the input is not a population code). This is the target for the input probes below.

In [ ]:
def collect_with_input(task, net, n_trials: int, device: str = "cpu"):
    """Like correlation_analysis.ipynb's collect(), but also keeps mean input response."""
    net.eval()
    all_r_hid, all_mu, all_sigma_sq, all_x_mean = [], [], [], []

    n_collected = 0
    with torch.no_grad():
        while n_collected < n_trials:
            batch = task.sample(include_internals=True)

            _, hidden = net(batch.input.to(device), return_hidden=True)
            mu = batch.opt_mean[:, :, 0].to(device)
            sigma_sq = batch.internals.opt_var[:, :, 0].to(device)
            # mean over the input-population dimension; if n_in == 1 this is
            # just the raw scalar measurement itself
            x_mean = batch.input.to(device).mean(dim=-1)

            all_r_hid.append(hidden.cpu())
            all_mu.append(mu.cpu())
            all_sigma_sq.append(sigma_sq.cpu())
            all_x_mean.append(x_mean.cpu())

            n_collected += batch.input.shape[0]

    return {
        "r_hid": torch.cat(all_r_hid, dim=0)[:n_trials],
        "mu": torch.cat(all_mu, dim=0)[:n_trials],
        "sigma_sq": torch.cat(all_sigma_sq, dim=0)[:n_trials],
        "x_mean": torch.cat(all_x_mean, dim=0)[:n_trials],
    }


device = "cpu"
checkpoint = torch.load(
    "checkpoints/kf_allgain_100batch.pt",
    map_location=device,
    weights_only=False,
)
net = KalmanRNN(
    n_in=checkpoint["config"]["n_in"],
    n_hid=checkpoint["config"]["n_hid"],
    n_out=1,
).to(device)
net.load_state_dict(checkpoint["state_dict"])

train_task = KalmanFilteringTask(batch_size=500, tr_cond="all_gains", seed=1000, device=device)
test_task = KalmanFilteringTask(batch_size=500, tr_cond="all_gains", seed=2000, device=device)

print("Collecting training set (with raw input)...")
train_data = collect_with_input(train_task, net, n_trials=5000, device=device)
print("Collecting held-out test set (with raw input)...")
test_data = collect_with_input(test_task, net, n_trials=2000, device=device)

torch.save(
    {"train": train_data, "test": test_data, "checkpoint_path": "checkpoints/kf_allgain_100batch.pt"},
    "saved_data/kf_allgain_100batch_dataset_with_input.pt",
)
print("Saved saved_data/kf_allgain_100batch_dataset_with_input.pt")


## Split trials and center per timestep

Same trial split / per-timestep centering convention as `decoder_analysis.ipynb`, applied here to `r_hid` and `x_mean` (means fit on train trials only).

In [ ]:
train_split = train_data
test_split = test_data
train_idx, val_idx = trial_train_validation_indices(train_split["r_hid"].shape[0])

train_trials = select_trials(train_split, train_idx)
val_trials = select_trials(train_split, val_idx)
test_trials = select_trials(test_split)

# hidden state: centered per timestep, mean fit on train trials
hidden_mean = train_trials["r_hid"].mean(dim=0)
r_hid_train = train_trials["r_hid"] - hidden_mean
r_hid_validation = val_trials["r_hid"] - hidden_mean
r_hid_test = test_trials["r_hid"] - hidden_mean

# raw input: centered per timestep, mean fit on train trials
x_train, x_mean_by_t = center_per_timestep(train_trials["x_mean"])
x_validation, _ = center_per_timestep(val_trials["x_mean"], train_mean=x_mean_by_t)
x_test, _ = center_per_timestep(test_trials["x_mean"], train_mean=x_mean_by_t)

# opt precision, for the info-plane comparison (recomputed here so this
# notebook is self-contained; matches decoder_analysis.ipynb's target)
precision_mean_by_t = opt_precision(train_trials["sigma_sq"]).mean(dim=0)
precision_train = opt_precision(train_trials["sigma_sq"]) - precision_mean_by_t
precision_validation = opt_precision(val_trials["sigma_sq"]) - precision_mean_by_t
precision_test = opt_precision(test_trials["sigma_sq"]) - precision_mean_by_t


## Sufficiency baseline: decode opt precision (for the info-plane axis)

In [ ]:
X_train_full = pool_timesteps(r_hid_train)
X_validation_full = pool_timesteps(r_hid_validation)
X_test_full = pool_timesteps(r_hid_test)

precision_decoder = LinearDecoder().fit(X_train_full, pool_timesteps(precision_train))
precision_r2 = precision_decoder.r2(X_test_full, pool_timesteps(precision_test))
print(f"Opt precision linear decoder test R^2: {precision_r2:.4f}")


## Minimality: decode raw input at several lags

For each lag `l`, fit a decoder `r_hid,t -> x_mean_{t-l}` and report held-out R^2 (linear and nonlinear). `burn_in` is set to the largest lag so every lag sees the same set of timesteps, matching the fIB paper's lag-probe convention.

In [ ]:
lags = [0, 1, 2, 4]
burn_in = max(lags)

lag_results = run_lag_sweep(
    r_hid_train, x_train,
    r_hid_validation, x_validation,
    r_hid_test, x_test,
    lags=lags,
    burn_in=burn_in,
    fit_nonlinear=True,
)

for r in lag_results:
    print(f"lag={r.lag:>2}  linear R^2={r.linear_r2:+.4f}  nonlinear R^2={r.nonlinear_r2:+.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

lags_arr = [r.lag for r in lag_results]
linear_r2s = [r.linear_r2 for r in lag_results]
nonlinear_r2s = [r.nonlinear_r2 for r in lag_results]

axes[0].plot(lags_arr, linear_r2s, "o-", label="linear", color="tab:blue")
axes[0].plot(lags_arr, nonlinear_r2s, "o-", label="nonlinear", color="tab:orange")
axes[0].set_xlabel("lag (timesteps back)")
axes[0].set_ylabel("input decoder test R^2")
axes[0].set_title("Raw input decodability vs lag")
axes[0].legend()

# information plane: input decodability (lag 0) vs posterior decodability
axes[1].scatter([linear_r2s[0]], [precision_r2], s=80, color="tab:purple")
axes[1].annotate(
    "this network",
    (linear_r2s[0], precision_r2),
    textcoords="offset points",
    xytext=(8, 8),
)
axes[1].set_xlabel("input decoder R^2 (lag 0)")
axes[1].set_ylabel("opt precision decoder R^2")
axes[1].set_title("Sufficiency vs minimality")
axes[1].set_xlim(-0.05, 1.05)
axes[1].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()


**Reading the right-hand plot:** high precision-R^2 with low input-R^2 (upper-left) is evidence of a compressed, minimal code. High precision-R^2 with high input-R^2 (upper-right) means the posterior is decodable but so is the raw input -- consistent with input re-representation rather than genuine compression, as Kalburge & Lengyel (2025) found for Kalman filtering. Note we have no copycat/PPC baseline here (unlike the fIB paper), so treat the absolute position as suggestive rather than calibrated; the *relative* pattern (does input R^2 drop off with lag while precision R^2 stays high?) is the more robust signal.